# Задание 6. Извлечение признаков из текста SMS

**Проект:** SMS Guard · **Автор:** Омаров М.Е., SIS-2304

План:
1. Стратифицированное разбиение 70 / 15 / 15 (`random_state=42`), индексы → `data/processed/sms_split.json`.
2. TF-IDF по словам (1–2-граммы) и по символам (`char_wb`, 3–5-граммы).
3. 8 ручных признаков.
4. Объединение через `ColumnTransformer` — код в `ml/features.py::build_vectorizer`.
5. `fit` только на обучающей выборке, `transform` — на валидационной и тестовой.
6. Размерности матриц и топ-15 признаков по среднему TF-IDF для каждого класса.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))
pd.set_option("display.width", 180)
pd.set_option("display.max_columns", 20)

from ml.features import (MANUAL_FEATURES, SPLIT_PATH, build_vectorizer, load_split,  # noqa: E402
                         save_split, split_dataset, top_tfidf_features)

df = pd.read_csv(PROJECT_ROOT / "data" / "processed" / "sms_clean.csv", keep_default_na=False)
print("Записей:", len(df))

Записей: 5135


## 1. Разбиение на выборки

Дубликаты уже удалены (задание 4), поэтому одно и то же сообщение не может попасть
в две выборки. Разбиение в два шага: 70% train / 30% temp, затем temp пополам на validation и test,
оба раза со стратификацией по метке.

In [2]:
split = split_dataset(df)
save_split(split, df)
parts = load_split(df)

summary = pd.DataFrame({
    name: {"Сообщений": len(part), "ham (0)": int((part["label"] == 0).sum()),
           "spam (1)": int((part["label"] == 1).sum()),
           "Доля spam, %": round(part["label"].mean() * 100, 2),
           "Доля от набора, %": round(len(part) / len(df) * 100, 1)}
    for name, part in parts.items()
}).T
print("Индексы сохранены в", SPLIT_PATH.relative_to(PROJECT_ROOT))
summary

Индексы сохранены в data/processed/sms_split.json


,Сообщений,ham (0),spam (1),"Доля spam, %","Доля от набора, %"
train,3594.0,3161.0,433.0,12.05,70.0
val,770.0,677.0,93.0,12.08,15.0
test,771.0,678.0,93.0,12.06,15.0


In [3]:
ids = {k: set(v) for k, v in split.items()}
assert not ids["train"] & ids["val"] and not ids["train"] & ids["test"] and not ids["val"] & ids["test"]
assert sum(len(v) for v in ids.values()) == len(df)
texts = {k: set(v["text_clean"]) for k, v in parts.items()}
print("Пересечений id между выборками: 0")
print("Одинаковых text_clean в train и test:", len(texts["train"] & texts["test"]))

Пересечений id между выборками: 0
Одинаковых text_clean в train и test: 0


## 2. Векторизатор

Параметры по ТЗ:

| Блок | Настройки |
| :--- | :--- |
| `word` — TF-IDF по словам | `ngram_range=(1, 2)`, `min_df=2`, `max_df=0.95`, `sublinear_tf=True`; токены — слова от 2 букв, спецтокены `<url>`, `<money>` …, знаки `!`, `?` |
| `char` — TF-IDF по символам | `analyzer='char_wb'`, `ngram_range=(3, 5)`; дополнительно `min_df=2`, `sublinear_tf=True`, чтобы отсечь единичные n-граммы. Спецтокены (`<num>` …) перед этим блоком удаляются: иначе они распадаются на n-граммы `<nu`, `num`, `um>` и дублируют словный блок |
| `manual` — ручные признаки | 8 признаков, масштабирование `MaxAbsScaler` в [0, 1] |

In [4]:
X_cols = ["text_raw", "text_clean"]
vectorizer = build_vectorizer()
vectorizer

,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('word', ...), ('char', ...), ...]"
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",1.0
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and ``feature

In [5]:
# fit — ТОЛЬКО на обучающей выборке
X_train = vectorizer.fit_transform(parts["train"][X_cols])
# validation и test — только transform
X_val = vectorizer.transform(parts["val"][X_cols])
X_test = vectorizer.transform(parts["test"][X_cols])
y_train, y_val, y_test = (parts[k]["label"].to_numpy() for k in ("train", "val", "test"))

blocks = {name: len(tr.get_feature_names_out()) for name, tr, _ in vectorizer.transformers_ if name != "remainder"}
shapes = pd.DataFrame({
    "Матрица": ["X_train", "X_val", "X_test"],
    "Строк (SMS)": [X_train.shape[0], X_val.shape[0], X_test.shape[0]],
    "Столбцов (признаков)": [X_train.shape[1], X_val.shape[1], X_test.shape[1]],
    "Ненулевых элементов": [X_train.nnz, X_val.nnz, X_test.nnz],
    "Заполненность, %": [round(m.nnz / (m.shape[0] * m.shape[1]) * 100, 3) for m in (X_train, X_val, X_test)],
})
print("Признаков по блокам:", blocks, "— всего", sum(blocks.values()))
assert X_train.shape[0] == len(parts["train"]) and X_val.shape[0] == len(parts["val"]) and X_test.shape[0] == len(parts["test"])
assert X_train.shape[1] == X_val.shape[1] == X_test.shape[1]
shapes

Признаков по блокам: {'word': 8943, 'char': 24543, 'manual': 8} — всего 33494


,Матрица,Строк (SMS),Столбцов (признаков),Ненулевых элементов,"Заполненность, %"
0,X_train,3594,33494,513443,0.427
1,X_val,770,33494,106179,0.412
2,X_test,771,33494,108187,0.419


Матрицы разреженные (`scipy.sparse.csr_matrix`): в каждом SMS встречается лишь малая
доля из ~33,5 тыс. возможных признаков, поэтому хранятся только ненулевые значения.

## 3. Топ-15 признаков по среднему TF-IDF для каждого класса (на обучающей выборке)

In [6]:
word_vec = vectorizer.named_transformers_["word"]
char_vec = vectorizer.named_transformers_["char"]
X_word = word_vec.transform(parts["train"]["text_clean"])
X_char = char_vec.transform(parts["train"]["text_clean"])

top_word = top_tfidf_features(word_vec, X_word, y_train)
top_char = top_tfidf_features(char_vec, X_char, y_train)


def show(top):
    return pd.DataFrame({
        "ham (0)": [f"{k} — {v:.3f}" for k, v in top[0].items()],
        "spam (1)": [f"{k} — {v:.3f}" for k, v in top[1].items()],
    }, index=range(1, 16))


print("Слова и биграммы:")
show(top_word)

Слова и биграммы:


,ham (0),spam (1)
1,you — 0.031,<num> — 0.077
2,? — 0.029,<money> — 0.063
3,to — 0.025,<phone> — 0.058
4,the — 0.020,to — 0.048
5,<num> — 0.020,! — 0.044
6,in — 0.019,call — 0.041
7,me — 0.018,your — 0.034
8,and — 0.017,to <num> — 0.033
9,it — 0.016,call <phone> — 0.032
10,my — 0.016,free — 0.031


In [7]:
print("Символьные n-граммы (пробел показан как «_»):")
show({k: v.rename(index=lambda s: s.replace(" ", "_")) for k, v in top_char.items()})

Символьные n-граммы (пробел показан как «_»):


,ham (0),spam (1)
1,_th — 0.019,_to — 0.024
2,_yo — 0.017,to_ — 0.024
3,_i_ — 0.017,_to_ — 0.023
4,you — 0.017,ur_ — 0.023
5,_you — 0.016,_ca — 0.022
6,ing — 0.016,call — 0.022
7,ng_ — 0.015,cal — 0.022
8,ing_ — 0.015,_!_ — 0.021
9,_to — 0.015,_call — 0.021
10,_?_ — 0.014,_cal — 0.021


**Что видно из топов.** У ham в топе слов — служебные слова (`you`, `to`, `the`) и «?»: стоп-слова
в этом варианте не удалялись, а личная переписка состоит в основном из них. У spam — спецтокены
`<num>`, `<money>`, `<phone>`, `<url>`, «!», призывы `call`, `txt`, `free`, `now` и биграммы
`call <phone>`, `to <num>` (отправить SMS на короткий номер). Влияние удаления стоп-слов
проверяется в задании 8. Среди символьных n-грамм spam — фрагменты `call`, `_fr` (free, from), `ur_` (сокращение ur / your),
`_!_`; у ham — частые фрагменты разговорной речи (`_you`, `ing_`, `_i_`). Символьные n-граммы
срабатывают и на искажённых написаниях, где словный блок слово не узнаёт.

## 4. Ручные признаки: средние значения по классам (обучающая выборка, до масштабирования)

In [8]:
from ml.features import manual_features  # noqa: E402

manual = pd.DataFrame(manual_features(parts["train"][X_cols]), columns=MANUAL_FEATURES)
manual["label"] = y_train
manual_means = manual.groupby("label").mean().T.round(3)
manual_means.columns = ["ham (0)", "spam (1)"]
manual_means["Описание"] = [
    "log(1 + длина в символах)", "Число слов", "Доля заглавных букв", "Число «!»",
    "Есть <url> (доля)", "Есть <money> (доля)", "Есть <phone> (доля)", "Сработавших групп словаря",
]
manual_means

,ham (0),spam (1),Описание
len_chars_log,4.042,4.898,log(1 + длина в символах)
n_words,14.186,23.776,Число слов
upper_ratio,0.076,0.162,Доля заглавных букв
n_exclaim,0.171,0.688,Число «!»
has_url,0.003,0.206,Есть <url> (доля)
has_money,0.004,0.543,Есть <money> (доля)
has_phone,0.001,0.543,Есть <phone> (доля)
n_rule_groups,0.038,0.739,Сработавших групп словаря


## 5. Пример: как одно сообщение превращается в признаки

In [9]:
from ml.features import make_input  # noqa: E402

example = make_input(["URGENT! You have won £900 prize. Call 09061701461 to claim"])
x = vectorizer.transform(example)
names = vectorizer.get_feature_names_out()
row = pd.Series(x.toarray().ravel(), index=names)
print("text_clean:", example["text_clean"][0])
print("Ненулевых признаков:", int((row > 0).sum()), "из", len(row))
print("\nСамые весомые признаки:")
row[row > 0].sort_values(ascending=False).head(12).round(3)

text_clean: urgent ! you have won <money> prize. call <phone> to claim
Ненулевых признаков: 104 из 33494

Самые весомые признаки:


manual__has_phone        1.000
manual__has_money        1.000
manual__len_chars_log    0.598
manual__n_rule_groups    0.500
word__prize call         0.325
word__! you              0.287
word__won <money>        0.270
word__<phone> to         0.259
word__to claim           0.256
word__urgent !           0.256
word__<money> prize      0.252
word__have won           0.251
dtype: float64

## 6. Назначение признаков

**TF-IDF** (term frequency — inverse document frequency) превращает текст в вектор чисел. Вес
термина в сообщении растёт, если термин часто встречается в этом сообщении (TF), и падает, если
он встречается почти во всех сообщениях набора (IDF). Поэтому служебные слова (`you`, `the`) получают
малый вес, а характерные (`prize`, `claim`, `<money>`) — большой. `sublinear_tf=True` заменяет
частоту на `1 + log(tf)`, чтобы трижды повторённое слово не весило втрое больше. `min_df=2`
отбрасывает термины, встретившиеся только в одном SMS (опечатки, имена), `max_df=0.95` — термины,
которые есть почти везде. Векторы нормируются (L2), поэтому длинное и короткое сообщение сравнимы.

**N-граммы слов** — последовательности из N соседних слов. Униграммы (`prize`) видят отдельные
слова, биграммы (`won prize`, `prize guaranteed`, `call <phone>`) — устойчивые обороты и порядок
слов, которых не видно по отдельным словам: «free» встречается и в личной переписке, а «free entry» —
почти только в спаме.

**Символьные N-граммы** (`char_wb`, 3–5 символов внутри границ слова) разбивают слово на
фрагменты: `prize` → ` pr`, `pri`, `riz`, `ize`, `ze `, `priz`, … Это даёт устойчивость к опечаткам,
сленгу и намеренным искажениям (`fr33`, `b@nk`, `w1n`): у искажённого слова остаются общие фрагменты
с исходным. Кроме того, символьные признаки работают с формами слов (`заблокирована`, `заблокирован`)
и с языками, для которых нет стеммера, — это важно для русского и казахского.

**Ручные признаки** переносят наблюдения EDA (задание 5) в модель напрямую: мошеннические SMS
длиннее, содержат больше «!» и заглавных букв, почти всегда — ссылку, телефон или сумму, а также
выражения из словаря социальной инженерии. Доля заглавных считается по исходному тексту, так как
в `text_clean` все буквы строчные.

**Защита от утечки данных.** Словари TF-IDF, IDF-веса и коэффициенты масштабирования вычисляются
методом `fit` только на обучающей выборке. Весь векторизатор входит в Pipeline модели (задание 7)
и сохраняется вместе с ней, поэтому веб-приложение преобразует текст точно так же, как при обучении.